In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import wandb
from datasets import get_dataset
import torch
from utils import save_run, load_run
from funcs import train_epoch, val_epoch, test_epoch
from weight_inits import init_timesformer
from torch.utils.data import DataLoader
from models.vivit_custom.utils.scheduler import WarmupCosineSchedule
from transforms import get_transforms, Div255, SplitEqually

from constants import get_classes
from transformers import TimesformerConfig, TimesformerForVideoClassification
from transformers import ViTForImageClassification
from pytorchvideo.transforms import MixUp
import torch
from torchvision.transforms.v2 import (
    ScaleJitter,
    RandAugment,
    RandomCrop,
    Compose,
    ColorJitter,
    RandomHorizontalFlip,
    Normalize,
    RandomApply,
    UniformTemporalSubsample,
    CenterCrop,
    Lambda,
)
from pytorchvideo.transforms import UniformTemporalSubsample, ApplyTransformToKey

In [ ]:
config = {
    "base_learning_rate": 0.001,
    "label_smoothing": 0.2,
    "epochs": 10,
    "frames": 8,
    "val_epochs": 1,
    "batch_size": 20,
    "weight_decay": 1e-5,
    "load_run": False,
    "init_vit": True,
    "vit_base_size": "tiny",
    "cosine_schedule": True,
    "augmentations": True,
    "advanced_augmentations": True,
    "mixup": True,
    "mixup_alpha": 0.2,
    "dataset": "ucf101",
}

In [ ]:
CLASSES_LIST = get_classes(config["dataset"])

ds_transform_train, ds_transform_valid, test_transform = get_transforms(config, CLASSES_LIST, config['dataset'])

dataset_train, dataset_valid, dataset_test, num_labels = get_dataset(
    config["dataset"], ds_transform_train, ds_transform_valid, test_transform, lazy=False
)

#training_loader = DataLoader(dataset=dataset_train, batch_size=8, shuffle=True)

In [ ]:
transform = Compose(
    transforms=[
        UniformTemporalSubsample(num_samples=config["frames"], temporal_dim=0),
        RandomHorizontalFlip(p=0.5),
        #ScaleJitter(target_size=(240, 320), scale_range=(0.85, 1.33)),
        RandomCrop(size=(224, 224), pad_if_needed=True, fill=128),
        RandomApply(p=1, transforms=[ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.1)]),
        # RandAugment(num_ops=2, magnitude=10, fill=128),
        Div255(),
        Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

In [ ]:
from utils import show_img_chw

In [ ]:
dataset_valid.transform = transform

In [ ]:
show_img_chw(dataset_valid[0][0][0], denormalize=True)

In [245]:
from vit_configs import get_vit_config
from transformers import TimesformerConfig, TimesformerForVideoClassification
from transformers import ViTForImageClassification
from weight_inits import init_timesformer


VIT_CONFIG = get_vit_config('tiny')
vit = ViTForImageClassification.from_pretrained(f"WinKawaks/vit-tiny-patch16-224")
timesformer_config = TimesformerConfig(num_frames=config["frames"], **VIT_CONFIG, num_labels=num_labels)
timesformer = TimesformerForVideoClassification(timesformer_config)


C:\Users\Wojtek\AppData\Roaming\Python\Python310\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [246]:
from models.switch.switch_timesformer import timesformer_to_switch, collect_route_stats

timesformer = timesformer_to_switch(timesformer, [0, 1], 4, timesformer_config)

In [247]:
timesformer.timesformer.encoder.layer[0].intermediate.experts[0].layer1.weight

Parameter containing:
tensor([[ 0.0075, -0.0487,  0.0227,  ..., -0.0675,  0.0072,  0.0174],
        [ 0.0699,  0.0286, -0.0542,  ...,  0.0270, -0.0383,  0.0295],
        [ 0.0700,  0.0114, -0.0349,  ..., -0.0445,  0.0345,  0.0446],
        ...,
        [-0.0241, -0.0326,  0.0061,  ..., -0.0329, -0.0588, -0.0111],
        [-0.0161,  0.0189, -0.0243,  ..., -0.0609, -0.0189,  0.0555],
        [ 0.0175,  0.0461,  0.0523,  ...,  0.0467,  0.0637,  0.0022]],
       requires_grad=True)

In [248]:
vit.vit.encoder.layer[0].intermediate.dense.weight

Parameter containing:
tensor([[ 0.0847, -0.0210,  0.0193,  ..., -0.0474,  0.0396,  0.0512],
        [ 0.1357,  0.0549,  0.0391,  ...,  0.0051,  0.0444, -0.0450],
        [ 0.0213, -0.0068,  0.0390,  ..., -0.0241, -0.0081,  0.0059],
        ...,
        [-0.0088, -0.0468, -0.0197,  ..., -0.0514,  0.0861, -0.0145],
        [ 0.0224,  0.0052, -0.0077,  ..., -0.0093, -0.0012, -0.0189],
        [-0.0514, -0.0132, -0.0101,  ..., -0.0412,  0.0252,  0.0471]],
       requires_grad=True)

In [249]:
timesformer = init_timesformer(timesformer, vit, timesformer_config.num_hidden_layers, True, [0, 1])

In [250]:
timesformer.timesformer.encoder.layer[0].intermediate.experts[0].layer1.weight

Parameter containing:
tensor([[ 0.0847, -0.0210,  0.0193,  ..., -0.0474,  0.0396,  0.0512],
        [ 0.1357,  0.0549,  0.0391,  ...,  0.0051,  0.0444, -0.0450],
        [ 0.0213, -0.0068,  0.0390,  ..., -0.0241, -0.0081,  0.0059],
        ...,
        [-0.0088, -0.0468, -0.0197,  ..., -0.0514,  0.0861, -0.0145],
        [ 0.0224,  0.0052, -0.0077,  ..., -0.0093, -0.0012, -0.0189],
        [-0.0514, -0.0132, -0.0101,  ..., -0.0412,  0.0252,  0.0471]],
       requires_grad=True)

In [251]:
vit.vit.encoder.layer[0].intermediate.dense.weight

Parameter containing:
tensor([[ 0.0847, -0.0210,  0.0193,  ..., -0.0474,  0.0396,  0.0512],
        [ 0.1357,  0.0549,  0.0391,  ...,  0.0051,  0.0444, -0.0450],
        [ 0.0213, -0.0068,  0.0390,  ..., -0.0241, -0.0081,  0.0059],
        ...,
        [-0.0088, -0.0468, -0.0197,  ..., -0.0514,  0.0861, -0.0145],
        [ 0.0224,  0.0052, -0.0077,  ..., -0.0093, -0.0012, -0.0189],
        [-0.0514, -0.0132, -0.0101,  ..., -0.0412,  0.0252,  0.0471]],
       requires_grad=True)

In [226]:
input = torch.rand((2, 8, 3,224,224))
with torch.no_grad():
    output= timesformer(input)

In [227]:
counts, route_prob, route_prob_max = collect_route_stats(timesformer, [0, 1])

In [230]:
route_prob_max

tensor([[0.2823, 0.4590, 0.4106,  ..., 0.3581, 0.5285, 0.3757],
        [0.4188, 0.3324, 0.3269,  ..., 0.3397, 0.4079, 0.3666]])